In [19]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col
from pyspark.sql import Window, functions as F


In [20]:
S3_BUCKET = "datalake-teste2"
S3_BRONZE = f"s3a://{S3_BUCKET}/bronze"
S3_SILVER = f"s3a://{S3_BUCKET}/silver"
S3_GOLD = f"s3a://{S3_BUCKET}/gold"

SPARK_MASTER = "local[*]"
SPARK_APP_NAME = "medallion-pipeline"

print(f"Bronze:  {S3_BRONZE}")
print(f"Silver:  {S3_SILVER}")
print(f"Gold:    {S3_GOLD}")

Bronze:  s3a://datalake-teste2/bronze
Silver:  s3a://datalake-teste2/silver
Gold:    s3a://datalake-teste2/gold


In [21]:
spark = SparkSession.builder \
    .appName(SPARK_APP_NAME) \
    .master(SPARK_MASTER) \
    .config("spark.hadoop.fs.s3a.endpoint", "http://localhost:4566") \
    .config("spark.hadoop.fs.s3a.access.key", "test") \
    .config("spark.hadoop.fs.s3a.secret.key", "test") \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .config("spark.jars.packages", "org.apache.hadoop:hadoop-aws:3.3.2,com.amazonaws:aws-java-sdk-bundle:1.12.261") \
    .getOrCreate()

In [22]:
df = spark.read.parquet(f"{S3_SILVER}/eventos_unificados")

print(f"\n📥 {df.count()} eventos lidos de eventos_unificados")


📥 36 eventos lidos de eventos_unificados


In [23]:
# Colunas que identificam o evento, presentes em todas as fontes.
CHAVES = ["transaction_datetime", "transaction_date", "purchase_id", "origem_evento"]


COLUNAS_POR_FONTE = {
    "purchase": ["buyer_id", "prod_item_id", "order_date", "release_date", "producer_id"],
    "product_item": ["product_id", "item_quantity", "purchase_value"],
    "purchase_extra_info": ["subsidiary"],
}


In [24]:
# --- 4.1 Forward fill por BLOCO DE FONTE --------------------------------
    # O jeito ingênuo — last(coluna, ignorenulls=True) — está ERRADO aqui,
    # porque NULL carrega dois significados incompatíveis na mesma coluna:
    #   a) "essa fonte não fala sobre esse campo"  -> deve herdar
    #   b) "essa fonte falou, e o valor é vazio"   -> NÃO deve herdar
    # Coluna a coluna os dois são indistinguíveis, e o ignorenulls pula os dois.
    # Resultado: o cancelamento do purchase_id=72 (release_date volta a NULL em
    # 10/05) seria ignorado e a compra contaria GMV para sempre.
    #
    # A correção é agrupar as colunas de cada fonte num struct que só existe
    # quando o evento veio daquela fonte. O struct inteiro é não-nulo sempre que
    # a fonte falou — mesmo que todos os campos dentro dele sejam NULL. Assim o
    # ignorenulls pula apenas eventos de OUTRAS fontes, nunca um evento real.
    
linha_do_tempo = Window.partitionBy("purchase_id").orderBy(
        F.col("transaction_datetime").asc(),
        F.col("hash_evento").asc(),
    ).rowsBetween(Window.unboundedPreceding, Window.currentRow)

for fonte, colunas in COLUNAS_POR_FONTE.items():
        bloco = F.when(F.col("origem_evento") == fonte, F.struct(*colunas))
        df = df.withColumn(
            f"_ultimo_{fonte}", F.last(bloco, ignorenulls=True).over(linha_do_tempo)
        )


In [25]:
# Expande os structs de volta em colunas planas. Só depois de TODOS estarem
# calculados, senão sobrescrever uma coluna afetaria o struct seguinte.
for fonte, colunas in COLUNAS_POR_FONTE.items():
    for coluna in colunas:
        df = df.withColumn(coluna, F.col(f"_ultimo_{fonte}.{coluna}"))
    df = df.drop(f"_ultimo_{fonte}")

In [26]:
# --- 4.2 Colapsa para o grão diário -------------------------------------
# Uma linha por (purchase_id, transaction_date): a última do dia, que já
# carrega o estado consolidado das 3 fontes até aquele momento.
dia = Window.partitionBy("purchase_id", "transaction_date")

In [27]:
# Quais fontes dispararam evento nesse dia — rastreabilidade diária.
    # Substitui origem_evento, que após o colapso só diria a última.
df = df.withColumn(
        "fontes_no_dia", F.sort_array(F.collect_set("origem_evento").over(dia))
    )

ultimo_do_dia = dia.orderBy(
        F.col("transaction_datetime").desc(),
        F.col("hash_evento").desc(),
    )

In [28]:
df = (df
      .withColumn("_rn", F.row_number().over(ultimo_do_dia))
      .filter(F.col("_rn") == 1)
      .drop("_rn", "origem_evento")
)

df.show()

+--------------------+-----------+--------+------------+----------+------------+-----------+----------+-------------+--------------+-------------+--------------------+----------------+--------------------+
|transaction_datetime|purchase_id|buyer_id|prod_item_id|order_date|release_date|producer_id|product_id|item_quantity|purchase_value|   subsidiary|         hash_evento|transaction_date|       fontes_no_dia|
+--------------------+-----------+--------+------------+----------+------------+-----------+----------+-------------+--------------+-------------+--------------------+----------------+--------------------+
| 2023-01-20 22:02:00|         55|   15947|           5|2023-01-20|  2023-01-20|     852852|    696969|           10|         50.00|         NULL|56bfb324c970e2e7b...|      2023-01-20|[product_item, pu...|
| 2023-01-23 00:05:00|         55|   15947|           5|2023-01-20|  2023-01-20|     852852|    696969|           10|         50.00|     nacional|0ed2b358ec2fe6113...|      202

In [29]:
df.drop("hash_evento").orderBy("purchase_id", "transaction_date").show(50, truncate=False)

+--------------------+-----------+--------+------------+----------+------------+-----------+----------+-------------+--------------+-------------+----------------+---------------------------------------------+
|transaction_datetime|purchase_id|buyer_id|prod_item_id|order_date|release_date|producer_id|product_id|item_quantity|purchase_value|subsidiary   |transaction_date|fontes_no_dia                                |
+--------------------+-----------+--------+------------+----------+------------+-----------+----------+-------------+--------------+-------------+----------------+---------------------------------------------+
|2023-01-20 22:02:00 |55         |15947   |5           |2023-01-20|2023-01-20  |852852     |696969    |10           |50.00         |NULL         |2023-01-20      |[product_item, purchase]                     |
|2023-01-23 00:05:00 |55         |15947   |5           |2023-01-20|2023-01-20  |852852     |696969    |10           |50.00         |nacional     |2023-01-23    

O que está fazendo

- Salvando no lake

In [ ]:
df.write.mode("overwrite").partitionBy("transaction_date").parquet(f"{S3_SILVER}/purchase_diario")